# 02. 베이스라인 vs KcELECTRA 성능 비교 시각화

**담당:** 경이  
**GPU 불필요:** 평가 결과 JSON을 불러와 시각화만 수행  

## 실행 전 체크리스트
- [x] `data/split_v1.csv` 존재 (`scripts/split_dataset.py` 실행 완료)
- [x] `src/classifier_simple.py`로 베이스라인 학습 완료 (`checkpoints/simple_tfidf_logreg.pkl`)
- [ ] `01_train_kcelectra.ipynb` 실행 완료 (`checkpoints/kcelectra-category/`)
- [ ] `scripts/evaluate_compare.py` 실행 완료 (`data/eval_results_*.json`)

## 이 노트북이 보여주는 것
1. Macro F1 막대 그래프 (두 모델 비교)
2. Per-class F1 레이더/막대 차트
3. Confusion Matrix 나란히 비교
4. 성능 향상 정리 표

In [ ]:
# ── 0. 설치 (필요 시 주석 해제) ──────────────────────────────
# !pip install matplotlib seaborn pandas scikit-learn --quiet

In [ ]:
# ── 1. 임포트 ─────────────────────────────────────────────────
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (Windows: 맑은고딕, macOS: AppleGothic)
import platform
if platform.system() == 'Windows':
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
else:
    matplotlib.rcParams['font.family'] = 'NanumGothic'  # Colab/Linux
matplotlib.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
LABELS   = ["일정", "준비물", "제출", "비용", "건강·안전", "기타"]

In [ ]:
# ── 2. 결과 파일 로드 ─────────────────────────────────────────
def load_result(fname: str) -> dict:
    path = DATA_DIR / fname
    if not path.exists():
        print(f"[경고] {path} 없음. 해당 모델 평가를 먼저 실행하세요.")
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)

simple_res     = load_result("eval_results_simple.json")
kcelectra_res  = load_result("eval_results_kcelectra.json")

results = [r for r in [simple_res, kcelectra_res] if r]
print(f"로드된 결과: {[r.get('model') for r in results]}")

In [ ]:
# ── 3. 베이스라인 직접 평가 (JSON 없을 경우 대비) ─────────────
if not simple_res:
    sys.path.insert(0, str(BASE_DIR))
    from src.classifier_simple import evaluate
    print("베이스라인 모델 직접 평가 중...")
    simple_res = evaluate(split="test")
    results = [simple_res]
    if kcelectra_res:
        results.append(kcelectra_res)

In [ ]:
# ── 4. Macro F1 비교 막대 그래프 ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

model_names = [r["model"] for r in results]
f1_values   = [r["macro_f1"] for r in results]
colors      = ["steelblue", "tomato"][:len(results)]

bars = ax.bar(model_names, f1_values, color=colors, width=0.4, edgecolor="black", linewidth=0.8)

# 값 레이블 표시
for bar, val in zip(bars, f1_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.4f}",
        ha="center", va="bottom", fontweight="bold", fontsize=12
    )

# 5% 향상 기준선 (베이스라인 기준)
if len(results) >= 2:
    threshold = simple_res["macro_f1"] + 0.05
    ax.axhline(threshold, color="green", linestyle="--", linewidth=1.2,
               label=f"채택 기준 (Simple+5%): {threshold:.4f}")
    ax.legend(loc="lower right")

ax.set_title("베이스라인 vs KcELECTRA — Macro F1 비교", fontsize=14)
ax.set_ylabel("Macro F1", fontsize=12)
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(DATA_DIR / "compare_macro_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5. Per-class F1 비교 막대 그래프 ─────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(LABELS))
width = 0.35
offsets = [-0.5, 0.5] if len(results) == 2 else [0]

for i, (res, color, label) in enumerate(zip(
    results,
    ["steelblue", "tomato"][:len(results)],
    [r["model"] for r in results],
)):
    per_class = res.get("per_class", {})
    f1s = [per_class.get(lab, {}).get("f1", 0.0) for lab in LABELS]
    offset = width * (i - (len(results) - 1) / 2)
    bars = ax.bar(x + offset, f1s, width=width, label=label, color=color,
                  edgecolor="black", linewidth=0.5)

ax.set_title("카테고리별 F1 비교", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=11)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(DATA_DIR / "compare_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 6. Confusion Matrix 나란히 비교 ──────────────────────────
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(8 * n_models, 6))
if n_models == 1:
    axes = [axes]

for ax, res in zip(axes, results):
    cm = np.array(res.get("confusion_matrix", [[]]))
    if cm.size == 0:
        continue
    sns.heatmap(
        cm, annot=True, fmt="d",
        xticklabels=LABELS, yticklabels=LABELS,
        cmap="Blues", ax=ax,
        linewidths=0.5, linecolor="gray",
    )
    ax.set_title(f"{res['model']} — Confusion Matrix", fontsize=13)
    ax.set_xlabel("예측", fontsize=11)
    ax.set_ylabel("실제", fontsize=11)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(DATA_DIR / "compare_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 7. 성능 요약 테이블 ───────────────────────────────────────
rows = []
for res in results:
    row = {
        "모델":           res["model"],
        "Macro F1":       res["macro_f1"],
        "Macro Precision": res["macro_precision"],
        "Macro Recall":   res["macro_recall"],
    }
    for label in LABELS:
        pc = res.get("per_class", {}).get(label, {})
        row[f"{label} F1"] = pc.get("f1", 0.0)
    rows.append(row)

summary_df = pd.DataFrame(rows)
print("\n── 성능 요약 ──")
display(summary_df)

if len(results) == 2:
    delta_f1 = kcelectra_res["macro_f1"] - simple_res["macro_f1"]
    print(f"\n Δ Macro F1 (KcELECTRA - Simple) = {delta_f1:+.4f}")
    if delta_f1 >= 0.05:
        print("  → KcELECTRA 5%+ 향상! 채택 권장 ✓")
    elif delta_f1 > 0:
        print("  → 소폭 향상. 서비스 안정성 고려해 선택.")
    else:
        print("  → Simple 유지 권장 (KcELECTRA 성능 이점 없음)")

# 요약 CSV 저장
summary_df.to_csv(DATA_DIR / "eval_comparison_summary.csv", index=False, encoding="utf-8-sig")
print(f"\n요약 저장 완료: {DATA_DIR / 'eval_comparison_summary.csv'}")

In [ ]:
# ── 8. 3개 지표 레이더 차트 (선택) ───────────────────────────
# 각 카테고리의 F1을 레이더로 시각화
if len(results) >= 1:
    angles = np.linspace(0, 2 * np.pi, len(LABELS), endpoint=False).tolist()
    angles += angles[:1]  # 닫기

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    palette = ["steelblue", "tomato"]

    for i, res in enumerate(results):
        per_class = res.get("per_class", {})
        vals = [per_class.get(lab, {}).get("f1", 0.0) for lab in LABELS]
        vals += vals[:1]
        ax.plot(angles, vals, color=palette[i], linewidth=2, label=res["model"])
        ax.fill(angles, vals, color=palette[i], alpha=0.2)

    ax.set_thetagrids(np.degrees(angles[:-1]), LABELS, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_title("카테고리별 F1 레이더 차트", pad=20, fontsize=14)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(DATA_DIR / "compare_radar_f1.png", dpi=150, bbox_inches="tight")
    plt.show()